In [ ]:
import numpy as np
import cv2
from skimage import io, color, morphology
from skimage.measure import label, regionprops
from skimage.filters import threshold_otsu
from skimage.color import rgb2hsv, rgb2gray
from skimage.segmentation import clear_border
import os

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except:
    HAS_MATPLOTLIB = False
    print('ADVERTENCIA: matplotlib no instalado, se mostraran datos numericos')

RUTA_IMAGEN = './image.png'
print(f'Imagen: {RUTA_IMAGEN}')
print(f'Existe: {os.path.exists(RUTA_IMAGEN)}')


# DETECCIÓN DE REGIONES Y OBJETOS CON COLORES

In [ ]:
# ============================================
# CARGAR IMAGEN
# ============================================
if not os.path.exists(RUTA_IMAGEN):
    print('Creando imagen de prueba...')
    np.random.seed(42)
    alto, ancho = 300, 400
    img = np.zeros((alto, ancho, 3), dtype=np.uint8)
    img[:] = [100, 100, 100]
    
    def circulo(img, cx, cy, r, color):
        y, x = np.ogrid[-r:r, -r:r]
        mask = x*x + y*y <= r*r
        img[cy-r:cy+r, cx-r:cx+r][mask] = color
        return img
    
    for i in range(6):
        cx, cy = np.random.randint(50, ancho-50), np.random.randint(50, alto-50)
        img = circulo(img, cx, cy, 30, [255, 0, 0])
    
    for i in range(5):
        cx, cy = np.random.randint(50, ancho-50), np.random.randint(50, alto-50)
        img = circulo(img, cx, cy, 25, [0, 0, 255])
    
    for i in range(7):
        cx, cy = np.random.randint(50, ancho-50), np.random.randint(50, alto-50)
        img = circulo(img, cx, cy, 28, [0, 255, 0])
    
    io.imsave(RUTA_IMAGEN, img)
    print(f'Imagen guardada en {RUTA_IMAGEN}')

img = io.imread(RUTA_IMAGEN)
if img.shape[-1] == 4:
    img = color.rgba2rgb(img)

print(f'Imagen cargada: {img.shape}')

# Convertir a diferentes espacios de color
img_gris = rgb2gray(img)
img_hsv = rgb2hsv(img)

print('Conversiones realizadas: RGB, GRAY, HSV')


In [ ]:
# ============================================
# DEFINIR RANGOS DE COLORES HSV
# ============================================
rangos_colores = {
    'ROJO':        {'h': (0.0, 0.04), 's': (0.5, 1.0), 'v': (0.5, 1.0)},
    'NARANJA':     {'h': (0.04, 0.08), 's': (0.5, 1.0), 'v': (0.5, 1.0)},
    'AMARILLO':    {'h': (0.08, 0.16), 's': (0.5, 1.0), 'v': (0.6, 1.0)},
    'VERDE':       {'h': (0.18, 0.38), 's': (0.4, 1.0), 'v': (0.4, 1.0)},
    'CYAN':        {'h': (0.45, 0.52), 's': (0.4, 1.0), 'v': (0.5, 1.0)},
    'AZUL':        {'h': (0.52, 0.65), 's': (0.4, 1.0), 'v': (0.4, 1.0)},
    'MAGENTA':     {'h': (0.75, 0.9), 's': (0.4, 1.0), 'v': (0.5, 1.0)},
    'BLANCO':      {'h': (0.0, 1.0), 's': (0.0, 0.15), 'v': (0.85, 1.0)},
    'NEGRO':       {'h': (0.0, 1.0), 's': (0.0, 0.25), 'v': (0.0, 0.15)}
}

# Crear mascaras para cada color
mascaras = {}
for nombre, rango in rangos_colores.items():
    h_mask = (img_hsv[:, :, 0] >= rango['h'][0]) & (img_hsv[:, :, 0] <= rango['h'][1])
    s_mask = (img_hsv[:, :, 1] >= rango['s'][0]) & (img_hsv[:, :, 1] <= rango['s'][1])
    v_mask = (img_hsv[:, :, 2] >= rango['v'][0]) & (img_hsv[:, :, 2] <= rango['v'][1])
    mascaras[nombre] = h_mask & s_mask & v_mask
    pixeles = np.sum(mascaras[nombre])
    print(f'  {nombre}: {pixeles} pixeles')


In [ ]:
# ============================================
# SEGMENTAR REGIONES POR UMBRAL
# ============================================
# Calcular umbral automatico con Otsu
umbral = threshold_otsu(img_gris)
print(f'Umbral Otsu: {umbral:.4f}')

# Crear imagen binaria
binaria = img_gris > umbral

# Limpiar bordes
binaria_limpia = clear_border(binaria)

# Aplicar operacion morfologica de apertura para reducir ruido
binaria_limpia = morphology.opening(binaria_limpia, morphology.disk(2))

# Rellenar huecos pequenos
binaria_limpia = morphology.remove_small_holes(binaria_limpia, area_threshold=30)

print(f'Region binaria creada')


In [ ]:
# ============================================
# ETIQUETAR REGIONES (OBJETOS)
# ============================================
etiquetas = label(binaria_limpia)
num_regiones = etiquetas.max()
print(f'Numero de regiones detectadas: {num_regiones}')

# Obtener propiedades de cada region
regiones = regionprops(etiquetas)

# Mostrar informacion de cada region
print('\n--- PROPIEDADES DE CADA REGION ---')
for i, reg in enumerate(regiones):
    print(f'Region {i+1}: Area={reg.area}, Perimetro={reg.perimeter:.1f}, Centro=({reg.centroid[1]:.1f}, {reg.centroid[0]:.1f})')


In [ ]:
# ============================================
# ANALIZAR COLOR DE CADA REGION
# ============================================
print('\n--- ANALISIS DE COLOR POR REGION ---')
print('-' * 70)

datos_regiones = []

for i, reg in enumerate(regiones):
    y1, x1, y2, x2 = reg.bbox
    
    # Contar pixeles de cada color en esta region
    conteo_colores = {}
    for nombre_color, mascara in mascaras.items():
        region_mascara = mascara[y1:y2, x1:x2]
        conteo_colores[nombre_color] = np.sum(region_mascara)
    
    # Encontrar color predominante
    color_predom = max(conteo_colores, key=conteo_colores.get)
    pixeles_predom = conteo_colores[color_predom]
    total_pixeles = (y2 - y1) * (x2 - x1)
    porcentaje = (pixeles_predom / total_pixeles) * 100 if total_pixeles > 0 else 0
    
    # Guardar datos
    datos_regiones.append({
        'region': i + 1,
        'area': reg.area,
        'centro_x': reg.centroid[1],
        'centro_y': reg.centroid[0],
        'color': color_predom,
        'porcentaje': porcentaje,
        'bbox': (y1, x1, y2, x2)
    })
    
    print(f'Region {i+1}: Color={color_predom:10s} Porcentaje={porcentaje:5.1f}% Area={reg.area}')

print('-' * 70)


In [ ]:
# ============================================
# RESUMEN TOTAL DE COLORES
# ============================================
conteo_final = {}
for datos in datos_regiones:
    color = datos['color']
    conteo_final[color] = conteo_final.get(color, 0) + 1

print('\n=== RESUMEN DE COLORES ===')
for color, count in sorted(conteo_final.items(), key=lambda x: -x[1]):
    print(f'  {color}: {count} regiones')

print(f'\nTotal regiones analizadas: {len(datos_regiones)}')


In [ ]:
# ============================================
# MOSTRAR VISUALIZACIONES
# ============================================
if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    # 1. Imagen original
    axes[0, 0].imshow(img)
    axes[0, 0].set_title(f'Original ({img.shape[0]}x{img.shape[1]})')
    axes[0, 0].axis('off')
    
    # 2. Region binaria
    axes[0, 1].imshow(binaria_limpia, cmap='gray')
    axes[0, 1].set_title(f'Regiones Binarias ({num_regiones})')
    axes[0, 1].axis('off')
    
    # 3. Etiquetas con colores
    axes[0, 2].imshow(etiquetas, cmap='nipy_spectral')
    axes[0, 2].set_title(f'Etiquetas ({num_regiones} regiones)')
    axes[0, 2].axis('off')
    
    # 4. Regiones con numero y color
    axes[0, 3].imshow(img)
    for datos in datos_regiones:
        y1, x1, y2, x2 = datos['bbox']
        # Dibujar rectangulo
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='yellow', linewidth=2)
        axes[0, 3].add_patch(rect)
        # Texto con region y color
        axes[0, 3].text(x1, y1-8, f'{datos["region"]}:{datos["color"]}', color='yellow', fontsize=9, fontweight='bold')
        # Centro
        axes[0, 3].plot(datos['centro_x'], datos['centro_y'], 'r+', markersize=10)
    axes[0, 3].set_title('Regiones Numeradas por Color')
    axes[0, 3].axis('off')
    
    # 5-8. Mascaras de colores principales
    colores_principales = ['ROJO', 'VERDE', 'AZUL', 'AMARILLO']
    for idx, color_nombre in enumerate(colores_principales):
        if color_nombre in mascaras:
            axes[1, idx].imshow(mascaras[color_nombre].astype(np.uint8) * 255, cmap='gray')
            pixeles = np.sum(mascaras[color_nombre])
            axes[1, idx].set_title(f'{color_nombre} ({pixeles} pixeles)')
            axes[1, idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('deteccion_regiones.png', dpi=100)
    print('\nGrafico guardado: deteccion_regiones.png')
    plt.close()
else:
    print('\nGrafico no disponible (matplotlib no instalado)')


In [ ]:
# ============================================
# TABLA COMPLETA DE DATOS
# ============================================
print('\n=== TABLA COMPLETA DE REGIONES ===')
print('-' * 80)
print(f'{"Region":>6} {"Area":>8} {"Centro X":>10} {"Centro Y":>10} {"Color":>12} {"Porcentaje":>10}')
print('-' * 80)
for datos in datos_regiones:
    print(f'{datos["region"]:>6} {datos["area"]:>8} {datos["centro_x"]:>10.1f} {datos["centro_y"]:>10.1f} {datos["color"]:>12} {datos["porcentaje"]:>10.1f}%')
print('-' * 80)
